In [18]:
import numpy as np
import pandas as pd
from pathlib import Path
import re

In [19]:
raw_path = Path.cwd().resolve().parent / 'data'/ 'raw'

In [20]:
snapshot_folder = raw_path / "2026-06-23"

calendar_path = snapshot_folder / "calendar.csv.gz"
listings_path = snapshot_folder / "listings.csv.gz"
reviews_path = snapshot_folder / "reviews.csv.gz"

calendar = pd.read_csv(calendar_path, encoding='utf-8', dtype={"listing_id": str})

listings = pd.read_csv(listings_path, encoding='utf-8', dtype={"id": str, "host_id": str, "listing_id": str})

reviews = pd.read_csv(reviews_path, encoding='utf-8', dtype={"id": str, "listing_id": str, "reviewer_id": str, "review_id": str})

In [21]:
print(len(calendar))
print(len(reviews))
print(reviews["Mẫu LLM"].notna().sum())
print(reviews["Nhãn tay"].notna().sum())

365000
5000
1000
100


In [22]:
def id_flags(data, prefix):
    """Tạo một cột True/False cho mỗi quy tắc kiểm tra ID."""
    ids = data['id']
    valid_id = ids.notna()
    repeated_id = valid_id & ids.duplicated()
    distinct_rows = data.loc[valid_id].drop_duplicates()
    conflicts = distinct_rows['id'].duplicated(keep=False)
    conflict_ids = distinct_rows.loc[conflicts, 'id']
    return pd.DataFrame({
        f'{prefix}_id_invalid': ~valid_id,
        f'{prefix}_duplicate_within': repeated_id,
        f'{prefix}_duplicate_conflict': ids.isin(conflict_ids),
    })

In [23]:
def clean_reviews(rv: pd.DataFrame, listings: pd.DataFrame):
    """Gắn vùng của listing vào review rồi kiểm tra ID và ngày."""
    before = rv.reset_index(drop=True).copy()
    before["comments"] = before["comments"].str.strip().str.replace(r"\r?<br\s*/?>", " ", regex=True, flags=re.I)
    before['review_date'] = pd.to_datetime(before["date"], format='%Y-%m-%d', errors='coerce')

    flags = id_flags(before, 'review')
    flags['review_date_missing'] = before["date"].isna()
    flags['review_date_invalid'] = before["date"].notna() & before['review_date'].isna()
    flags['review_outside_listings'] = ~(before['listing_id'].isin(listings["id"]))
    return before, flags

In [36]:
cleaned_rv, flags = clean_reviews(reviews, listings) 

In [37]:
rv_to_label = cleaned_rv[cleaned_rv["Nhãn tay"].str.strip().str.lower()=='x'][["id", "comments"]].reset_index(drop=True)
samples_path = Path.cwd().resolve().parent / 'data'/ 'samples'
samples_path.mkdir(parents=True, exist_ok=True)

In [38]:
nhan_tay = pd.read_csv(samples_path / "rv_to_label.csv", encoding='utf-8', dtype={"id": str})
nhan_tay.loc[rv_to_label["comments"] != nhan_tay["comments"]]

,id,comments,general_sentiment,accessibility,noise


In [27]:
import sys
import yaml
from pathlib import Path

project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)
from src import llm

config_path = Path.cwd().resolve().parent / "configs" / "porto.yaml"
with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)

In [28]:
nhan_tay = pd.read_csv(samples_path / "rv_to_label.csv", encoding='utf-8')
labeled = llm.llm(nhan_tay, config_path)

In [29]:
llm = labeled.add_prefix("llm_")
llm = llm.rename(columns={"llm_comments": "comments"})
so_sanh = nhan_tay.merge(llm, how='left', on="comments")
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]].sum()

id                                                     2346009829199378738
comments                 As other reviews mentioned, there’s a funky sm...
general_sentiment                                       MixedMixedNegative
accessibility                      Not mentionedNot mentionedNot mentioned
noise                              Not mentionedNot mentionedNot mentioned
llm_general_sentiment                                PositivePositiveMixed
llm_accessibility                  Not mentionedNot mentionedNot mentioned
llm_noise                          Not mentionedNot mentionedNot mentioned
dtype: object

In [30]:
so_sanh[so_sanh["accessibility"] != so_sanh["llm_accessibility"]]

,id,comments,general_sentiment,accessibility,noise,llm_general_sentiment,llm_accessibility,llm_noise
9,394739523347319693,Fantastic place that’s perfectly located to ex...,Positive,Highly accessible,Not mentioned,Positive,Not mentioned,Not mentioned
50,884598596754501546,"Pedro was a great host, he is kind and friendl...",Positive,Highly accessible,Not mentioned,Positive,Not mentioned,Not mentioned
52,536784433,"Lovely studio, with just what’s needed for a g...",Positive,Not mentioned,Not mentioned,Positive,Highly accessible,Not mentioned
67,627317499556472048,Everything was great! Communication was spot o...,Positive,Highly accessible,Not mentioned,Positive,Not mentioned,Not mentioned
69,1149890992320615625,"This is a beautiful flat, with amazing views. ...",Positive,Not mentioned,Not mentioned,Positive,Poor access,Not mentioned
88,1494138362470243968,The flat and surroundings were perfect to visi...,Positive,Highly accessible,Quiet,Positive,Not mentioned,Quiet


In [31]:
# so_sanh[so_sanh["accessibility"] != so_sanh["llm_accessibility"]].loc[9, "comments"]

In [32]:
so_sanh[so_sanh["noise"] != so_sanh["llm_noise"]]

,id,comments,general_sentiment,accessibility,noise,llm_general_sentiment,llm_accessibility,llm_noise
17,1421723267915957445,Hugo’s place was very easy to find and he prov...,Positive,Highly accessible,Acceptable,Positive,Highly accessible,Quiet


In [33]:
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]]

,id,comments,general_sentiment,accessibility,noise,llm_general_sentiment,llm_accessibility,llm_noise
31,811461050575532798,"As other reviews mentioned, there’s a funky sm...",Mixed,Not mentioned,Not mentioned,Positive,Not mentioned,Not mentioned
60,916581544482959773,everything is OK. Couch is not new with some s...,Mixed,Not mentioned,Not mentioned,Positive,Not mentioned,Not mentioned
78,617967234140886167,"The apartment itself was great, however, the c...",Negative,Not mentioned,Not mentioned,Mixed,Not mentioned,Not mentioned


In [35]:
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]].iloc[0, 1]

'As other reviews mentioned, there’s a funky smell throughout the place…And we found it a little bit cold in the apartment. But honestly I don’t know if that can be avoided in a historical home like this one - The charm outweighs the negatives. There are heaters in each room but they can’t really keep upbeat the high ceilings. We would stay here again, just maybe bring warmer clothes next time.'